In [ ]:
import logging
import os
import time
import h5py
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import optax
from scipy.sparse.linalg import eigsh
import pickle
import numpy as np
from NES_VMC_V1 import (
    NESTotalAnsatz,
    NESTotalAnsatz_stable,
    create_single_machine_gauge_fixed,
    Ham_Psi_scaled,
    flatten_batched_pytree,
    NESFermionHopRule,
    ravel_pytree,
)
from NES_VMC_tool import create_gauge_reset_total_machines,NES_loss_energy_stable_gauge,\
    nes_vmc_gradient_stable_gauge,make_grad_fn_gauge,make_qgt_fn_gauge,make_gauge_fn
from He_ccpvD import SINGLE_SIZE, ha, hi_ext, ext_edges, K, Hatree_Fock,hi,E_fcis
import logging

In [ ]:
N_CHAINS = 16
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 10
N_ITER = 10

time_str = time.strftime("%y-%m-%d-%H-%M")   # 输出文件名统一时间前
Natural_Grad = True
clip_norm = 20.0
lr = 0.1
qgt_diag_shift = 0.1
RESET_PERIOD = 10          # 每隔多少步做一次 gauge reset（30 轮内 3 次）
SAVE_INTERVAL = 20         # 每多少步保存/追加一次 pickle
HISTORY_FILE = f"./data/{time_str}_history_natural_gradient_He_atom_K4.pkl"
os.makedirs("./data", exist_ok=True)

# ====================== 模型与采样器 ======================
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=SINGLE_SIZE,
    n_states=K,
    hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)

g_current = jnp.zeros(K, dtype=jnp.complex64)   # 全局列规范 g，非训练参数，动态传入 jit
(
    total_machine,
    total_matrix_machine,
    total_max_machine,
    total_matrix_machine_raw,
    total_graphdef,
    total_params,
) = create_gauge_reset_total_machines(total_ansatz, Hatree_Fock)
single_machine_list = [
    create_single_machine_gauge_fixed(ansatz, Hatree_Fock)[0]
    for ansatz in total_ansatz.single_ansatz_list
]
grad_fn = make_grad_fn_gauge(
    ha,
    total_matrix_machine,
    total_max_machine,
    total_machine,
    single_machine_list,
)
qgt_fn = make_qgt_fn_gauge(total_machine)

gauge_fn, col_mean_fn = make_gauge_fn(total_ansatz, Hatree_Fock)

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=SWEEP_SIZE,
)

# FCI 精确参考能量
exact_eigvals = E_fcis
# ====================== 优化器 ======================
optimizer = optax.chain(
    optax.clip_by_global_norm(clip_norm),
    optax.sgd(learning_rate=lr),
)
opt_state = optimizer.init(total_params)
sampler_rng = jax.random.PRNGKey(21)

def sample_machine(params, sigma):
    """双参数封装：采样只需要 |Ψ|² 的转移率，全局列规范 g 是常数平移，不影响 ratio。
    闭包每次调用读取 g_current 的最新值（g 作为动态参数传入 jit）。"""
    return total_machine(params, sigma, g_current)

sampler_state = nes_sampler.init_state(sample_machine, total_params, sampler_rng)

In [ ]:
file_path = './data/26-09-04-03-06_history_natural_gradient_He_atom_K4.pkl'
with open (file_path, "rb") as f:
    history = pickle.load(f)


In [ ]:
history.keys()

In [ ]:
from NES_VMC_V1 import SingleStateAnsatz
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=SINGLE_SIZE,
    n_states=K,
    hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)
single_ansatz = SingleStateAnsatz(
    n_spin_orbitals=SINGLE_SIZE,
    hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)


In [ ]:
GraphDef, total_params = nnx.split(total_ansatz)
single_GraphDef, single_params = nnx.split(single_ansatz)


In [ ]:
total_params['single_ansatz_list'][0]

In [ ]:
single_ansatz = nnx.merge(single_GraphDef, single_params) #已经加载好训练好的θ
ha.to_dense().shape #（25,25） 哈密顿量
hi.all_states()[0] #（25,10） 所有状态

In [ ]:
ha.get_conn(hi.all_states()[0]) 

In [ ]:
hi.all_states()[0]

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

# ============================================================
# 1. 构造完整波函数向量 psi_vec, shape = (25,)
# ============================================================
all_states = hi.all_states()          # (25, 10)
N_states = all_states.shape[0]

# NNX 模块前向: 对每个组态求值
# 注意: NetKet 的 ansatz 通常输出 log|psi| (实数) 或 log psi (复数)
# 用 vmap 批量求值, 比循环快
log_psi_vec = jax.vmap(lambda x: single_ansatz(x))(all_states)  # (25,)

# 如果 ansatz 输出是 log 振幅, 还原为 psi
# (NetKet 默认模型输出 log_psi, 实值对应 log|psi|, 复值对应 log psi)
psi_vec = jnp.exp(log_psi_vec)        # (25,)

# ============================================================
# 2. 精确能量 E = <psi|H|psi> / <psi|psi>
# ============================================================
H_dense = ha.to_dense()               # (25, 25)

# 归一化 (防止数值溢出, 虽然 25 维不会有问题)
psi_vec = psi_vec / jnp.linalg.norm(psi_vec)

E_num = jnp.vdot(psi_vec, H_dense @ psi_vec).real
E_den = jnp.vdot(psi_vec, psi_vec).real   # 归一化后 = 1
E_variational = E_num 

print(f"变分基态能量 E = {E_variational:.8f}")

# ============================================================
# 3. 精确对角化基态能量 (对照校验)
# ============================================================
eigenvalues = jnp.linalg.eigvalsh(H_dense)
E_exact_gs = eigenvalues[0]
print(f"精确对角化基态能量 E0 = {E_exact_gs:.8f}")
print(f"能量误差 ΔE = {E_variational - E_exact_gs:.2e}")

# ============================================================
# 4. (可选) 用局域能量方式复算, 验证一致性
#    E_loc(x) = sum_{x'} H_{x,x'} psi(x') / psi(x)
# ============================================================
# H @ psi 给出每个组态上的 (H|psi>)(x)
Hpsi = H_dense @ psi_vec              # (25,)
E_loc = Hpsi / psi_vec                # (25,) 局域能量
E_from_loc = jnp.mean(E_loc * jnp.abs(psi_vec)**2 / jnp.sum(jnp.abs(psi_vec)**2)).real
print(f"局域能量加权平均 E = {E_from_loc:.8f}  (应与上面一致)")
E_fcis #array([-2.88759483, -1.40116328, -0.952151  , -0.38469286])

In [ ]:
from He_ccpvD import E_fcis

E_fcis #array([-2.88759483, -1.40116328, -0.952151  , -0.38469286])


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

# ============================================================
# 1. 构造完整波函数向量 psi_vec, shape = (25,)
# ============================================================
all_states = hi.all_states()          # (25, 10)
N_states = all_states.shape[0]

# NNX 模块前向: 对每个组态求值
# 注意: NetKet 的 ansatz 通常输出 log|psi| (实数) 或 log psi (复数)
# 用 vmap 批量求值, 比循环快
log_psi_vec = jax.vmap(lambda x: single_ansatz(x))(all_states)  # (25,)

# 如果 ansatz 输出是 log 振幅, 还原为 psi
# (NetKet 默认模型输出 log_psi, 实值对应 log|psi|, 复值对应 log psi)
psi_vec = jnp.exp(log_psi_vec)        # (25,)

# ============================================================
# 2. 精确能量 E = <psi|H|psi> / <psi|psi>
# ============================================================
H_dense = ha.to_dense()               # (25, 25)

# 归一化 (防止数值溢出, 虽然 25 维不会有问题)
psi_vec = psi_vec / jnp.linalg.norm(psi_vec)

E_num = jnp.vdot(psi_vec, H_dense @ psi_vec).real
E_den = jnp.vdot(psi_vec, psi_vec).real   # 归一化后 = 1
E_variational = E_num / E_den

print(f"变分基态能量 E = {E_variational:.8f}")

# ============================================================
# 3. 精确对角化基态能量 (对照校验)
# ============================================================
eigenvalues = jnp.linalg.eigvalsh(H_dense)
E_exact_gs = eigenvalues[0]
print(f"精确对角化基态能量 E0 = {E_exact_gs:.8f}")
print(f"能量误差 ΔE = {E_variational - E_exact_gs:.2e}")

# ============================================================
# 4. (可选) 用局域能量方式复算, 验证一致性
#    E_loc(x) = sum_{x'} H_{x,x'} psi(x') / psi(x)
# ============================================================
# H @ psi 给出每个组态上的 (H|psi>)(x)
Hpsi = H_dense @ psi_vec              # (25,)
E_loc = Hpsi / psi_vec                # (25,) 局域能量
E_from_loc = jnp.mean(E_loc * jnp.abs(psi_vec)**2 / jnp.sum(jnp.abs(psi_vec)**2)).real
print(f"局域能量加权平均 E = {E_from_loc:.8f}  (应与上面一致)")
